# Carichi e combinazioni — esempio completo

Calcolo delle azioni e delle combinazioni SLU/SLE per un edificio residenziale secondo NTC18.

**Dati geometrici**:
- Edificio residenziale (categoria A)
- Zona vento: 3, altitudine: 250 m s.l.m.
- Zona neve: II, altitudine: 250 m s.l.m.
- Periodo di ritorno di riferimento: 50 anni

In [ ]:
from pyntc.actions.loads import unit_weight, variable_load, partition_equivalent_load
from pyntc.actions.wind import (
    wind_base_velocity,
    wind_reference_velocity,
    wind_kinetic_pressure,
    wind_exposure_coefficient,
    wind_pressure,
)
from pyntc.actions.snow import snow_ground_load, snow_shape_coefficient, snow_roof_load
from pyntc.actions.combinations import (
    combination_coefficients,
    partial_safety_factors,
    slu_combination,
    sle_characteristic_combination,
    sle_quasi_permanent_combination,
)

## 1. Carichi permanenti — NTC18 §3.1

In [ ]:
# Peso proprio struttura (solaio in c.a.)
gamma_ca = unit_weight("calcestruzzo_armato")  # kN/m³
spessore_solaio = 0.20  # m
G1 = gamma_ca * spessore_solaio
print(f"Peso proprio solaio G1 = {G1:.2f} kN/m²")

# Carichi permanenti portati (pavimento + massetto + partizioni)
G2_pavimento = 1.0   # kN/m²  pavimento + massetto
G2_partizioni = partition_equivalent_load(weight_per_m=1.5)  # kN/m²
G2 = G2_pavimento + G2_partizioni
print(f"Partizioni equivalenti = {G2_partizioni:.2f} kN/m²")
print(f"Carichi permanenti portati G2 = {G2:.2f} kN/m²")

## 2. Carichi variabili — NTC18 §3.1, Tab. 3.1.II

In [ ]:
# Categoria A — residenziale
qk, Qk, Hk = variable_load("A")
print(f"Carico distribuito qk = {qk:.1f} kN/m²")
print(f"Carico concentrato Qk = {Qk:.1f} kN")
print(f"Carico orizzontale Hk = {Hk:.1f} kN/m")

## 3. Azione del vento — NTC18 §3.3

In [ ]:
zona = 3
altitudine = 250.0  # m s.l.m.
z = 10.0            # altezza dal suolo [m]
cat_esposizione = 3 # categoria di esposizione

v_b = wind_base_velocity(zona, altitudine)
v_r = wind_reference_velocity(zona, altitudine, return_period=50)
q_b = wind_kinetic_pressure(v_r)
c_e = wind_exposure_coefficient(z, cat_esposizione)

# Pressione su parete sopravento (cp = +0.8) e sottovento (cp = -0.5)
p_sopravento = wind_pressure(q_b, c_e, c_p=+0.8)
p_sottovento = wind_pressure(q_b, c_e, c_p=-0.5)
Qw = p_sopravento - p_sottovento  # pressione netta

print(f"Velocità base v_b      = {v_b:.2f} m/s")
print(f"Velocità riferimento v_r = {v_r:.2f} m/s")
print(f"Pressione cinetica q_b  = {q_b:.3f} kN/m²")
print(f"Coeff. esposizione c_e  = {c_e:.3f}")
print(f"Pressione netta vento   = {Qw:.3f} kN/m²")

## 4. Azione della neve — NTC18 §3.4

In [ ]:
zona_neve = "II"
q_sk = snow_ground_load(zona_neve, altitudine)

# Copertura piana (angolo 0°)
mu = snow_shape_coefficient(alpha=0.0)
C_e = 1.0  # esposizione normale
C_t = 1.0  # coefficiente termico
qs = snow_roof_load(q_sk, mu, C_e, C_t)

print(f"Carico neve al suolo q_sk = {q_sk:.3f} kN/m²")
print(f"Coeff. forma μ            = {mu:.2f}")
print(f"Carico neve in copertura  = {qs:.3f} kN/m²")

## 5. Coefficienti di combinazione — NTC18 §2.5.3, Tab. 2.5.I

In [ ]:
psi0_A, psi1_A, psi2_A = combination_coefficients("A")         # residenziale
psi0_w, psi1_w, psi2_w = combination_coefficients("wind")      # vento
psi0_s, psi1_s, psi2_s = combination_coefficients("snow_low")  # neve ≤1000 m

print(f"ψ residenziale  ψ0={psi0_A}, ψ1={psi1_A}, ψ2={psi2_A}")
print(f"ψ vento         ψ0={psi0_w}, ψ1={psi1_w}, ψ2={psi2_w}")
print(f"ψ neve          ψ0={psi0_s}, ψ1={psi1_s}, ψ2={psi2_s}")

## 6. Combinazioni SLU — NTC18 §2.5.3, Formula [2.5.1]

In [ ]:
gamma_G1_sfav, gamma_G2_sfav, gamma_Q_sfav = partial_safety_factors("A1", favorable=False)

# SLU — azione variabile principale: residenziale
# Azioni accompagnatrici: vento e neve con ψ0
E_slu = slu_combination(
    G1=G1, G2=G2,
    Q_leading=qk,
    Q_others=[Qw * psi0_w, qs * psi0_s],
    approach="A1",
    favorable=False,
)
print(f"SLU (residenziale principale) = {E_slu:.3f} kN/m²")

# SLU — azione variabile principale: vento
E_slu_vento = slu_combination(
    G1=G1, G2=G2,
    Q_leading=Qw,
    Q_others=[qk * psi0_A, qs * psi0_s],
    approach="A1",
    favorable=False,
)
print(f"SLU (vento principale)        = {E_slu_vento:.3f} kN/m²")

E_slu_max = max(E_slu, E_slu_vento)
print(f"\nSLU di progetto (inviluppo)   = {E_slu_max:.3f} kN/m²")

## 7. Combinazioni SLE — NTC18 §2.5.3, Formule [2.5.3]–[2.5.5]

In [ ]:
E_sle_car = sle_characteristic_combination(
    G1=G1, G2=G2,
    Q_leading=qk,
    Q_others=[Qw * psi0_w, qs * psi0_s],
)

E_sle_qp = sle_quasi_permanent_combination(
    G1=G1, G2=G2,
    Q_values=[qk, Qw, qs],
    psi2_values=[psi2_A, psi2_w, psi2_s],
)

print(f"SLE caratteristica    = {E_sle_car:.3f} kN/m²")
print(f"SLE quasi-permanente  = {E_sle_qp:.3f} kN/m²")

## Riepilogo

In [ ]:
print("=" * 45)
print("RIEPILOGO AZIONI E COMBINAZIONI")
print("=" * 45)
print(f"G1  (peso proprio)       = {G1:.2f}  kN/m²")
print(f"G2  (permanenti portati) = {G2:.2f}  kN/m²")
print(f"qk  (residenziale)       = {qk:.2f}  kN/m²")
print(f"Qw  (vento netto)        = {Qw:.3f} kN/m²")
print(f"qs  (neve)               = {qs:.3f} kN/m²")
print("-" * 45)
print(f"SLU di progetto          = {E_slu_max:.3f} kN/m²")
print(f"SLE caratteristica       = {E_sle_car:.3f} kN/m²")
print(f"SLE quasi-permanente     = {E_sle_qp:.3f} kN/m²")
print("=" * 45)